# Phase 1: Data Preparation & Cleaning (Python / Pandas)

**Objective:** Ingest raw semi-structured JSON and tabular CSV datasets, perform data cleaning, handle missing values and formatting inconsistencies, enforce optimal data types, and export standardized datasets for SQL Server ingestion.

## 1. Environment Setup & Workspace Configuration

In [1]:
import os
import pandas as pd

if os.path.basename(os.getcwd()) in ['Advance', 'notebooks']:
    base_path = os.path.join('..', 'archive')
    output_base_path = os.path.join('..', 'cleaned_data')
else:
    base_path = os.path.join('archive')
    output_base_path = os.path.join('cleaned_data')

# Create the output folder
os.makedirs(output_base_path, exist_ok=True)

print(f"Raw data directory: {base_path}")
print(f"Cleaned data output directory: {output_base_path}")

Raw data directory: ..\archive
Cleaned data output directory: ..\cleaned_data


## 2. Ingestion & Flattening of Semi-Structured JSON Files

In [2]:
mcc_codes = pd.read_json(os.path.join(base_path, 'mcc_codes.json'), typ='series').reset_index()
mcc_codes.columns = ['mcc_code', 'description']

train_fraud = pd.read_json(os.path.join(base_path, 'train_fraud_labels.json')).reset_index()
train_fraud.columns = ['transaction_id', 'is_fraud']

### 2.1 Inspection: Merchant Category Codes (`mcc_codes.json`)

In [3]:
mcc_codes.info()

<class 'pandas.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   mcc_code     109 non-null    int64
 1   description  109 non-null    str  
dtypes: int64(1), str(1)
memory usage: 1.8 KB


In [4]:
mcc_codes.head(5)

,mcc_code,description
0,5812,Eating Places and Restaurants
1,5541,Service Stations
2,7996,"Amusement Parks, Carnivals, Circuses"
3,5411,"Grocery Stores, Supermarkets"
4,4784,Tolls and Bridge Fees


### 2.2 Inspection: Target Fraud Labels (`train_fraud_labels.json`)

In [5]:
train_fraud.info()

<class 'pandas.DataFrame'>
RangeIndex: 8914963 entries, 0 to 8914962
Data columns (total 2 columns):
 #   Column          Dtype
---  ------          -----
 0   transaction_id  int64
 1   is_fraud        str  
dtypes: int64(1), str(1)
memory usage: 136.0 MB


In [6]:
train_fraud.head(5)

,transaction_id,is_fraud
0,10649266,No
1,23410063,No
2,9316588,No
3,12478022,No
4,9558530,No


## 3. Standardization & Cleaning of Relational CSV Files

### 3.1 Data Loading

In [7]:
cards_data = pd.read_csv(os.path.join(base_path, 'cards_data.csv'))
transactions_data = pd.read_csv(os.path.join(base_path, 'transactions_data.csv'))
users_data = pd.read_csv(os.path.join(base_path, 'users_data.csv'))

---

### 3.2 Cleaning & Processing: Card Metadata (`cards_data.csv`)

#### 3.2.1 Initial Data Exploration & Structure Inspection

In [8]:
# 1. Dataset info
cards_data.info()

# 2. Check for missing values in each column
print("\n--- Missing Values Count ---")
display(cards_data.isna().sum())

# 3. Check for duplicated rows
print(f"\nTotal duplicate rows: {cards_data.duplicated().sum()}")

# 4. Head
cards_data.head()

<class 'pandas.DataFrame'>
RangeIndex: 6146 entries, 0 to 6145
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   id                     6146 non-null   int64
 1   client_id              6146 non-null   int64
 2   card_brand             6146 non-null   str  
 3   card_type              6146 non-null   str  
 4   card_number            6146 non-null   int64
 5   expires                6146 non-null   str  
 6   cvv                    6146 non-null   int64
 7   has_chip               6146 non-null   str  
 8   num_cards_issued       6146 non-null   int64
 9   credit_limit           6146 non-null   str  
 10  acct_open_date         6146 non-null   str  
 11  year_pin_last_changed  6146 non-null   int64
 12  card_on_dark_web       6146 non-null   str  
dtypes: int64(6), str(7)
memory usage: 624.3 KB

--- Missing Values Count ---


id                       0
client_id                0
card_brand               0
card_type                0
card_number              0
expires                  0
cvv                      0
has_chip                 0
num_cards_issued         0
credit_limit             0
acct_open_date           0
year_pin_last_changed    0
card_on_dark_web         0
dtype: int64


Total duplicate rows: 0


,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


#### 3.2.2 Currency Standardisation (`credit_limit`)

In [9]:
# Remove '$' and commas, then cast to float
cards_data['credit_limit'] = (
    cards_data['credit_limit']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

# Verify summary statistics for credit limit
cards_data['credit_limit'].describe()

count      6146.000000
mean      14347.493980
std       12014.463884
min           0.000000
25%        7042.750000
50%       12592.500000
75%       19156.500000
max      151223.000000
Name: credit_limit, dtype: float64

#### 3.2.3 Boolean Encoding (`has_chip`, `card_on_dark_web`)

In [10]:
# Convert 'has_chip' (YES/NO -> True/False)
cards_data['has_chip'] = cards_data['has_chip'].str.strip().str.upper() == 'YES'

# Convert 'card_on_dark_web' (Yes/No -> True/False)
cards_data['card_on_dark_web'] = cards_data['card_on_dark_web'].str.strip().str.capitalize() == 'Yes'

# Check value counts to confirm conversion
print("has_chip distribution:\n", cards_data['has_chip'].value_counts())
print("\ncard_on_dark_web distribution:\n", cards_data['card_on_dark_web'].value_counts())

has_chip distribution:
 has_chip
True     5500
False     646
Name: count, dtype: int64

card_on_dark_web distribution:
 card_on_dark_web
False    6146
Name: count, dtype: int64


#### 3.2.4 Datetime Parsing & Formatting (`acct_open_date`, `expires`)

In [11]:
# 'acct_open_date' starts at the 1st of the month
cards_data['acct_open_date'] = pd.to_datetime(cards_data['acct_open_date'], format='%m/%Y')

# 'expires' ends on the last day of the month (e.g., 2022-12-31)
cards_data['expires'] = (
    pd.to_datetime(cards_data['expires'], format='%m/%Y') 
    + pd.offsets.MonthEnd(1)
)

In [12]:
cards_data.head()

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,2022-12-31,623,True,2,24295.0,2002-09-01,2008,False
1,2731,825,Visa,Debit,4956965974959986,2020-12-31,393,True,2,21968.0,2014-04-01,2014,False
2,3701,825,Visa,Debit,4582313478255491,2024-02-29,719,True,2,46414.0,2003-07-01,2004,False
3,42,825,Visa,Credit,4879494103069057,2024-08-31,693,False,1,12400.0,2003-01-01,2012,False
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,2009-03-31,75,True,1,28.0,2008-09-01,2009,False


#### 3.2.5 Final Data Type & Structure Validation

In [13]:
cards_data.info()
cards_data.head()

<class 'pandas.DataFrame'>
RangeIndex: 6146 entries, 0 to 6145
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id                     6146 non-null   int64         
 1   client_id              6146 non-null   int64         
 2   card_brand             6146 non-null   str           
 3   card_type              6146 non-null   str           
 4   card_number            6146 non-null   int64         
 5   expires                6146 non-null   datetime64[us]
 6   cvv                    6146 non-null   int64         
 7   has_chip               6146 non-null   bool          
 8   num_cards_issued       6146 non-null   int64         
 9   credit_limit           6146 non-null   float64       
 10  acct_open_date         6146 non-null   datetime64[us]
 11  year_pin_last_changed  6146 non-null   int64         
 12  card_on_dark_web       6146 non-null   bool          
dtypes: bool(2), da

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,2022-12-31,623,True,2,24295.0,2002-09-01,2008,False
1,2731,825,Visa,Debit,4956965974959986,2020-12-31,393,True,2,21968.0,2014-04-01,2014,False
2,3701,825,Visa,Debit,4582313478255491,2024-02-29,719,True,2,46414.0,2003-07-01,2004,False
3,42,825,Visa,Credit,4879494103069057,2024-08-31,693,False,1,12400.0,2003-01-01,2012,False
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,2009-03-31,75,True,1,28.0,2008-09-01,2009,False


---

### 3.3 Cleaning & Processing: Transaction Records (`transactions_data.csv`)

#### 3.3.1 Initial Exploration & Structure Inspection

In [14]:
# 1. Dataset info
transactions_data.info()

# 2. Check missing values
print("\n--- Missing Values Count ---")
display(transactions_data.isna().sum())

# 3. Preview first 5 rows
transactions_data.head()

<class 'pandas.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 12 columns):
 #   Column          Dtype  
---  ------          -----  
 0   id              int64  
 1   date            str    
 2   client_id       int64  
 3   card_id         int64  
 4   amount          str    
 5   use_chip        str    
 6   merchant_id     int64  
 7   merchant_city   str    
 8   merchant_state  str    
 9   zip             float64
 10  mcc             int64  
 11  errors          str    
dtypes: float64(1), int64(5), str(6)
memory usage: 1.2 GB

--- Missing Values Count ---


id                       0
date                     0
client_id                0
card_id                  0
amount                   0
use_chip                 0
merchant_id              0
merchant_city            0
merchant_state     1563700
zip                1652706
mcc                      0
errors            13094522
dtype: int64

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


#### 3.3.2 Financial Amount Normalization (`amount`)

In [15]:
# Clean amount column and cast to float
transactions_data['amount'] = (
    transactions_data['amount']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

# Check distribution (min, max, mean)
transactions_data['amount'].describe()

count    1.330592e+07
mean     4.297604e+01
std      8.165575e+01
min     -5.000000e+02
25%      8.930000e+00
50%      2.899000e+01
75%      6.371000e+01
max      6.820200e+03
Name: amount, dtype: float64

#### 3.3.3 Datetime Conversion (`date`)

In [16]:
# Convert to datetime
transactions_data['date'] = pd.to_datetime(transactions_data['date'], format='%Y-%m-%d %H:%M:%S')

# Optional: verify date range
print(f"Date range: from {transactions_data['date'].min()} to {transactions_data['date'].max()}")

Date range: from 2010-01-01 00:01:00 to 2019-10-31 23:59:00


#### 3.3.4 Missing Value Imputation (`errors`)

In [17]:
# Fill missing error values with 'No Error'
transactions_data['errors'] = transactions_data['errors'].fillna('No Error')

# Inspect the most common transaction errors
print(transactions_data['errors'].value_counts().head(10))

errors
No Error                                 13094522
Insufficient Balance                       130902
Bad PIN                                     32119
Technical Glitch                            26271
Bad Card Number                              7767
Bad Expiration                               6161
Bad CVV                                      6106
Bad Zipcode                                  1126
Bad PIN,Insufficient Balance                  293
Insufficient Balance,Technical Glitch         243
Name: count, dtype: int64


#### 3.3.5 Memory Optimization via Categorical Data Types

In [18]:
# Convert repetitive string columns to 'category' to save memory
categorical_cols = ['use_chip', 'merchant_state', 'errors']

for col in categorical_cols:
    transactions_data[col] = transactions_data[col].astype('category')

#### 3.3.6 Zip Code Formatting & Standardisation (`zip`)

In [19]:
# Convert numeric ZIPs to 5-digit strings (e.g., 58523.0 -> "58523")
transactions_data['zip'] = (
    transactions_data['zip']
    .fillna(-1)
    .astype(int)
    .astype(str)
    .replace('-1', pd.NA)
    .str.zfill(5)
)

#### 3.3.7 Final Validation

In [20]:
transactions_data.info()
transactions_data.head()

<class 'pandas.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 12 columns):
 #   Column          Dtype         
---  ------          -----         
 0   id              int64         
 1   date            datetime64[us]
 2   client_id       int64         
 3   card_id         int64         
 4   amount          float64       
 5   use_chip        category      
 6   merchant_id     int64         
 7   merchant_city   str           
 8   merchant_state  category      
 9   zip             str           
 10  mcc             int64         
 11  errors          category      
dtypes: category(3), datetime64[us](1), float64(1), int64(5), str(2)
memory usage: 964.4 MB


,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,59935,Beulah,ND,58523,5499,No Error
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722,5311,No Error
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084,4829,No Error
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307,4829,No Error
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776,5813,No Error


---

### 3.4 Cleaning & Processing: Customer Demographics (`users_data.csv`)

#### 3.4.1 Exploration

In [21]:
# 1. Dataset info
users_data.info()

# 2. Check for missing values
print("\n--- Missing Values Count ---")
display(users_data.isna().sum())

# 3. Check for duplicate users
print(f"\nTotal duplicate rows: {users_data.duplicated().sum()}")

# 4. Data head
users_data.head()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 2000 non-null   int64  
 1   current_age        2000 non-null   int64  
 2   retirement_age     2000 non-null   int64  
 3   birth_year         2000 non-null   int64  
 4   birth_month        2000 non-null   int64  
 5   gender             2000 non-null   str    
 6   address            2000 non-null   str    
 7   latitude           2000 non-null   float64
 8   longitude          2000 non-null   float64
 9   per_capita_income  2000 non-null   str    
 10  yearly_income      2000 non-null   str    
 11  total_debt         2000 non-null   str    
 12  credit_score       2000 non-null   int64  
 13  num_credit_cards   2000 non-null   int64  
dtypes: float64(2), int64(7), str(5)
memory usage: 218.9 KB

--- Missing Values Count ---


id                   0
current_age          0
retirement_age       0
birth_year           0
birth_month          0
gender               0
address              0
latitude             0
longitude            0
per_capita_income    0
yearly_income        0
total_debt           0
credit_score         0
num_credit_cards     0
dtype: int64


Total duplicate rows: 0


,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


#### 3.4.2 Financial Currency Standardisation

In [22]:
# List of financial columns with '$' signs
currency_cols = ['per_capita_income', 'yearly_income', 'total_debt']

for col in currency_cols:
    users_data[col] = (
        users_data[col]
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
    )

# Summary statistics for the financial figures
users_data[currency_cols].describe()

,per_capita_income,yearly_income,total_debt
count,2000.000000,2000.000000,2000.000000
mean,23141.928000,45715.882000,63709.694000
std,11324.137358,22992.615456,52254.453421
min,0.000000,1.000000,0.000000
25%,16824.500000,32818.500000,23986.750000
50%,20581.000000,40744.500000,58251.000000
75%,26286.000000,52698.500000,89070.500000
max,163145.000000,307018.000000,516263.000000


#### 3.4.3 Feature Engineering: Birth Date Mapping (`birth_date`)

In [23]:
# Create a birth_date column (YYYY-MM)
users_data['birth_date'] = pd.to_datetime(
    users_data['birth_year'].astype(str) + '-' + users_data['birth_month'].astype(str).str.zfill(2) + '-01'
).dt.to_period('M')

# Preview the result
users_data[['birth_year', 'birth_month', 'birth_date', 'current_age']].head()

,birth_year,birth_month,birth_date,current_age
0,1966,11,1966-11,53
1,1966,12,1966-12,53
2,1938,11,1938-11,81
3,1957,1,1957-01,63
4,1976,9,1976-09,43


#### 3.4.4 Categorical Encoding (`gender`)

In [24]:
# Convert gender to category
users_data['gender'] = users_data['gender'].astype('category')

# Check distribution
print("Gender distribution:\n", users_data['gender'].value_counts())

Gender distribution:
 gender
Female    1016
Male       984
Name: count, dtype: int64


#### 3.4.5 Domain Validation & Sanity Checks

In [25]:
# 1. Credit score range (typical FICO scores are 300 - 850)
print(f"Credit Score Range: {users_data['credit_score'].min()} - {users_data['credit_score'].max()}")

# 2. Age distribution
print(f"Age Range: {users_data['current_age'].min()} - {users_data['current_age'].max()}")

# 3. Quick check for negative incomes or debts
invalid_finances = users_data[(users_data['yearly_income'] < 0) | (users_data['total_debt'] < 0)]
print(f"Users with negative income/debt: {len(invalid_finances)}")

Credit Score Range: 480 - 850
Age Range: 18 - 101
Users with negative income/debt: 0


#### 3.4.6 Final Validation

In [26]:
users_data.info()
users_data.head()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype    
---  ------             --------------  -----    
 0   id                 2000 non-null   int64    
 1   current_age        2000 non-null   int64    
 2   retirement_age     2000 non-null   int64    
 3   birth_year         2000 non-null   int64    
 4   birth_month        2000 non-null   int64    
 5   gender             2000 non-null   category 
 6   address            2000 non-null   str      
 7   latitude           2000 non-null   float64  
 8   longitude          2000 non-null   float64  
 9   per_capita_income  2000 non-null   float64  
 10  yearly_income      2000 non-null   float64  
 11  total_debt         2000 non-null   float64  
 12  credit_score       2000 non-null   int64    
 13  num_credit_cards   2000 non-null   int64    
 14  birth_date         2000 non-null   period[M]
dtypes: category(1), float64(5), int64(7), period[M](1

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,birth_date
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,29278.0,59696.0,127613.0,787,5,1966-11
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,37891.0,77254.0,191349.0,701,5,1966-12
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,22681.0,33483.0,196.0,698,5,1938-11
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,163145.0,249925.0,202328.0,722,4,1957-01
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,53797.0,109687.0,183855.0,675,1,1976-09


## 4. Exporting Cleaned Datasets

In [27]:
os.makedirs(output_base_path, exist_ok=True)

print(f"Output directory ready at: {output_base_path}")

Output directory ready at: ..\cleaned_data


In [28]:
# 1. Cards Data
cards_data.to_csv(os.path.join(output_base_path, 'cards_data_clean.csv'), index=False)
print("✓ cards_data_clean.csv saved")

# 2. Users Data
users_data.to_csv(os.path.join(output_base_path, 'users_data_clean.csv'), index=False)
print("✓ users_data_clean.csv saved")

# 3. MCC Codes
mcc_codes.to_csv(os.path.join(output_base_path, 'mcc_codes_clean.csv'), index=False)
print("✓ mcc_codes_clean.csv saved")

# 4. Train Fraud Labels
train_fraud.to_csv(os.path.join(output_base_path, 'train_fraud_clean.csv'), index=False)
print("✓ train_fraud_clean.csv saved")

# 5. Transactions Data
transactions_data.to_csv(os.path.join(output_base_path, 'transactions_data_clean.csv'), index=False)
print("✓ transactions_data_clean.csv saved")

✓ cards_data_clean.csv saved
✓ users_data_clean.csv saved
✓ mcc_codes_clean.csv saved
✓ train_fraud_clean.csv saved
✓ transactions_data_clean.csv saved
